# Deploy MCP Servers on OpenShift

This notebook verifies existing MCP server deployments and fixes any missing resources.

**Servers to deploy:**

| # | Server | Type | Air-gapped | Purpose |
|---|--------|------|:----------:|---------|
| 1 | Context7 | External API | No | Library documentation lookup |
| 2 | SearXNG | Self-hosted meta-engine | No | Web search & page fetching |
| 3 | Code Sandbox | Local | Yes | Secure Python/Bash/Node execution |
| 4 | Codebase Search | Custom (AI) | Yes | Semantic code search over internal repo |
| 5 | Repo Docs | Custom (AI) | Yes | Internal documentation Q&A |

> Servers 4-5 use **sentence-transformers** for local embeddings — no external LLM dependency.
> In air-gapped environments, skip servers 1-2 (they require internet).

## 1. Verify Cluster Access

In [1]:
%%bash
echo "Cluster: $(oc whoami --show-server)"
echo "User: $(oc whoami)"
echo ""
echo "Apps domain (for Route URLs):"
oc get ingresses.config cluster -o jsonpath='{.spec.domain}'
echo ""

Cluster: https://api.openshift-cluster.sandbox3031.opentlc.com:6443
User: admin

Apps domain (for Route URLs):
apps.openshift-cluster.sandbox3031.opentlc.com


## 2. Verify Namespace

All MCP servers deploy into the `mcp-servers` namespace.

In [2]:
%%bash
if oc get ns mcp-servers &>/dev/null; then
    echo "Namespace exists:"
    oc get ns mcp-servers -o jsonpath='{.metadata.name} (status: {.status.phase})'
    echo ""
else
    echo "ERROR: Namespace mcp-servers does not exist."
    echo "Run: oc apply -f manifests/00-namespace-secret.yaml"
    exit 1
fi

Namespace exists:
mcp-servers (status: Active)


## 3. Verify Server 1 — Context7 (Library Documentation)

Context7 provides up-to-date library documentation for AI agents.
Requires outbound internet to call Upstash's API.

**Tools:** `resolve-library-id`, `get-library-docs`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | No (skip this cell) |

In [3]:
%%bash
DEPLOY="mcp-context7"
NS="mcp-servers"

if oc get deployment/$DEPLOY -n $NS &>/dev/null; then
    STATUS=$(oc get deployment/$DEPLOY -n $NS -o jsonpath='{.status.conditions[?(@.type=="Available")].status}')
    if [ "$STATUS" = "True" ]; then
        echo "$DEPLOY is already running."
        echo ""
        echo "Route URL:"
        oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
        echo ""
        exit 0
    fi
    echo "$DEPLOY exists but is not ready. Re-applying manifest..."
fi

echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://mcp.context7.com/mcp)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "Internet reachable - deploying Context7..."
    oc apply -f manifests/02-context7.yaml
    echo ""
    oc wait --for=condition=available deployment/$DEPLOY -n $NS --timeout=120s 2>/dev/null \
        && echo "Context7 Pod ready" \
        || echo "Pod still starting... check: oc get pods -n $NS"
    echo ""
    echo "Route URL:"
    oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "No internet access (HTTP $HTTP_CODE) - skipping Context7."
    echo "This server requires outbound connectivity to Upstash API."
fi

Checking outbound internet access...
Internet reachable - deploying Context7...
deployment.apps/mcp-context7 created
service/mcp-context7 created
route.route.openshift.io/mcp-context7 created

deployment.apps/mcp-context7 condition met
Context7 Pod ready

Route URL:
https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp


## 4. Verify Server 2 — SearXNG (Web Search)

SearXNG is a self-hosted meta-search engine bundled as a single Docker image.
No external API keys, no rate limits — searches via multiple upstream engines.

**Tools:** `search-web`, `fetch-web`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | No (skip this cell) |

In [4]:
%%bash
DEPLOY="mcp-searxng"
NS="mcp-servers"

if oc get deployment/$DEPLOY -n $NS &>/dev/null; then
    STATUS=$(oc get deployment/$DEPLOY -n $NS -o jsonpath='{.status.conditions[?(@.type=="Available")].status}')
    if [ "$STATUS" = "True" ]; then
        echo "$DEPLOY is already running."
        echo ""
        echo "Route URL:"
        oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
        echo ""
        exit 0
    fi
    echo "$DEPLOY exists but is not ready. Re-applying manifest..."
fi

echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://www.google.com)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "301" ] || [ "$HTTP_CODE" = "302" ]; then
    echo "Internet reachable - deploying SearXNG..."
    oc apply -f manifests/04-searxng.yaml
    echo ""
    oc wait --for=condition=available deployment/$DEPLOY -n $NS --timeout=180s 2>/dev/null \
        && echo "SearXNG Pod ready" \
        || echo "Pod still starting... check: oc get pods -n $NS"
    echo ""
    echo "Route URL:"
    oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "No internet access (HTTP $HTTP_CODE) - skipping SearXNG."
    echo "This server requires outbound connectivity for search results."
fi

mcp-searxng is already running.

Route URL:


Error from server (NotFound): routes.route.openshift.io "mcp-searxng" not found


## 5. Verify Server 3 — Code Sandbox (Secure Execution)

Secure code execution sandbox. Runs Python, Bash, and Node.js in an isolated workspace.
No external dependencies — works fully offline.

**Tools:** `execute_code`, `read_file`, `write_file`, `list_files`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | Yes |

In [5]:
%%bash
DEPLOY="mcp-code-sandbox"
NS="mcp-servers"

if oc get deployment/$DEPLOY -n $NS &>/dev/null; then
    STATUS=$(oc get deployment/$DEPLOY -n $NS -o jsonpath='{.status.conditions[?(@.type=="Available")].status}')
    if [ "$STATUS" = "True" ]; then
        echo "$DEPLOY is already running."
        echo ""
        echo "Route URL:"
        oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
        echo ""
        exit 0
    fi
    echo "$DEPLOY exists but is not ready (likely missing ConfigMap mcp-code-sandbox-src)."
    echo "Applying manifest to create missing resources..."
fi

oc apply -f manifests/03-code-sandbox.yaml

echo ""
oc wait --for=condition=available deployment/$DEPLOY -n $NS --timeout=180s 2>/dev/null \
    && echo "Code Sandbox ready" \
    || echo "Pod still starting..."

echo ""
echo "Route URL:"
oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
echo ""

configmap/mcp-code-sandbox-src created
deployment.apps/mcp-code-sandbox created
service/mcp-code-sandbox created
route.route.openshift.io/mcp-code-sandbox created

deployment.apps/mcp-code-sandbox condition met
Code Sandbox ready

Route URL:
https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp


## 6. Custom AI Servers — Build Images

Servers 4 and 5 use **sentence-transformers** for local semantic search.
We build container images with the embedding model pre-downloaded (avoids runtime download).

**Build time**: ~5-8 minutes (downloads `all-MiniLM-L6-v2` model, ~80MB)

In [6]:
%%bash
echo "=== Building mcp-codebase-search image ==="
echo ""

if ! oc get bc mcp-codebase-search -n mcp-servers &>/dev/null; then
    oc new-build --binary --strategy=docker --name=mcp-codebase-search -n mcp-servers 2>&1
    echo ""
fi

echo "Starting build (this takes ~5 min)..."
oc start-build mcp-codebase-search \
    --from-dir=./mcp-codebase-search \
    -n mcp-servers \
    --follow --wait

=== Building mcp-codebase-search image ===

    * A Docker build using binary input will be created
      * The resulting image will be pushed to image stream tag "mcp-codebase-search:latest"
      * A binary build was created, use 'oc start-build --from-dir' to trigger a new build

--> Creating resources with label build=mcp-codebase-search ...
    imagestream.image.openshift.io "mcp-codebase-search" created
    buildconfig.build.openshift.io "mcp-codebase-search" created
--> Success

Starting build (this takes ~5 min)...


Uploading directory "mcp-codebase-search" as binary input for the build ...

Uploading finished


build.build.openshift.io/mcp-codebase-search-1 started
Receiving source from STDIN as archive ...
time="2026-07-21T07:35:08Z" level=info msg="Not using native diff for overlay, this may cause degraded performance for building images: kernel has CONFIG_OVERLAY_FS_REDIRECT_DIR enabled"
I0721 07:35:08.928491       1 defaults.go:112] Defaulting to storage driver "overlay" with options [mountopt=metacopy=on].
Caching blobs under "/var/cache/blobs".

Pulling image python:3.11-slim ...
Resolving "python" using unqualified-search registries (/etc/containers/registries.conf)
Trying to pull registry.redhat.io/python:3.11-slim...
Trying to pull registry.access.redhat.com/python:3.11-slim...
Trying to pull quay.io/python:3.11-slim...
Trying to pull docker.io/library/python:3.11-slim...
Getting image source signatures
Copying blob sha256:c89b9f64c028c19ba92e195d7589d914b9cd1fc69f3c3dfeb931e93876ac2064
Copying blob sha256:6b265b8eae4a26a263eb55546aa430c1eef37032a7cd86393d98c6b779f28319
Copying blob 

In [7]:
%%bash
echo "=== Building mcp-repo-docs image ==="
echo ""

if ! oc get bc mcp-repo-docs -n mcp-servers &>/dev/null; then
    oc new-build --binary --strategy=docker --name=mcp-repo-docs -n mcp-servers 2>&1
    echo ""
fi

echo "Starting build (this takes ~5 min)..."
oc start-build mcp-repo-docs \
    --from-dir=./mcp-repo-docs \
    -n mcp-servers \
    --follow --wait

=== Building mcp-repo-docs image ===

    * A Docker build using binary input will be created
      * The resulting image will be pushed to image stream tag "mcp-repo-docs:latest"
      * A binary build was created, use 'oc start-build --from-dir' to trigger a new build

--> Creating resources with label build=mcp-repo-docs ...
    imagestream.image.openshift.io "mcp-repo-docs" created
    buildconfig.build.openshift.io "mcp-repo-docs" created
--> Success

Starting build (this takes ~5 min)...


Uploading directory "mcp-repo-docs" as binary input for the build ...

Uploading finished


build.build.openshift.io/mcp-repo-docs-1 started
Receiving source from STDIN as archive ...
time="2026-07-21T07:39:45Z" level=info msg="Not using native diff for overlay, this may cause degraded performance for building images: kernel has CONFIG_OVERLAY_FS_REDIRECT_DIR enabled"
I0721 07:39:45.139161       1 defaults.go:112] Defaulting to storage driver "overlay" with options [mountopt=metacopy=on].
Caching blobs under "/var/cache/blobs".

Pulling image python:3.11-slim ...
Resolving "python" using unqualified-search registries (/etc/containers/registries.conf)
Trying to pull registry.redhat.io/python:3.11-slim...
Trying to pull registry.access.redhat.com/python:3.11-slim...
Trying to pull quay.io/python:3.11-slim...
Trying to pull docker.io/library/python:3.11-slim...
Getting image source signatures
Copying blob sha256:062e450697faa5f02a3a74eba9864ee4d79bc9cfbd65769fc6cdff2c05c6a053
Copying blob sha256:6b265b8eae4a26a263eb55546aa430c1eef37032a7cd86393d98c6b779f28319
Copying blob sha256

In [8]:
%%bash
echo "Build results:"
echo ""
oc get builds -n mcp-servers --sort-by=.metadata.creationTimestamp | tail -5
echo ""
echo "Image streams:"
oc get is -n mcp-servers -o custom-columns=NAME:.metadata.name,TAGS:.status.tags[0].tag,UPDATED:.status.tags[0].items[0].created

Build results:

NAME                    TYPE     FROM             STATUS     STARTED         DURATION
mcp-codebase-search-1   Docker   Binary@9248b7c   Complete   9 minutes ago   4m32s
mcp-repo-docs-1         Docker   Binary@9248b7c   Complete   4 minutes ago   4m40s

Image streams:
NAME                  TAGS     UPDATED
mcp-codebase-search   latest   2026-07-21T07:39:35Z
mcp-repo-docs         latest   2026-07-21T07:44:20Z


## 7. Create Data ConfigMaps

Servers 4 and 5 perform semantic search over *your* code and docs.
They need the actual files mounted into their pods — we package them as ConfigMaps.

- **cafe-source-code**: Python source files from `cafe-order-system` (indexed by Codebase Search)
- **cafe-docs**: Internal documentation — architecture, API guide, runbook, etc. (indexed by Repo Docs)

In [9]:
%%bash
CAFE_APP_DIR="../0_setup/apps/cafe-order-system"

echo "=== Creating cafe-source-code ConfigMap ==="

oc delete configmap cafe-source-code -n mcp-servers 2>/dev/null

oc create configmap cafe-source-code -n mcp-servers \
    --from-file=main.py=${CAFE_APP_DIR}/app/main.py \
    --from-file=config.py=${CAFE_APP_DIR}/app/config.py \
    --from-file=database.py=${CAFE_APP_DIR}/app/database.py \
    --from-file=models.py=${CAFE_APP_DIR}/app/models.py \
    --from-file=schemas.py=${CAFE_APP_DIR}/app/schemas.py \
    --from-file=routes_menu.py=${CAFE_APP_DIR}/app/routes/menu.py \
    --from-file=routes_orders.py=${CAFE_APP_DIR}/app/routes/orders.py \
    --from-file=routes_customers.py=${CAFE_APP_DIR}/app/routes/customers.py \
    --from-file=services_order.py=${CAFE_APP_DIR}/app/services/order_service.py \
    --from-file=services_inventory.py=${CAFE_APP_DIR}/app/services/inventory_service.py \
    --from-file=requirements.txt=${CAFE_APP_DIR}/requirements.txt \
    --from-file=Dockerfile=${CAFE_APP_DIR}/Dockerfile

echo ""
echo "=== Creating cafe-docs ConfigMap ==="

oc delete configmap cafe-docs -n mcp-servers 2>/dev/null

oc create configmap cafe-docs -n mcp-servers \
    --from-file=architecture.md=${CAFE_APP_DIR}/docs/architecture.md \
    --from-file=api-guide.md=${CAFE_APP_DIR}/docs/api-guide.md \
    --from-file=onboarding.md=${CAFE_APP_DIR}/docs/onboarding.md \
    --from-file=security-policy.md=${CAFE_APP_DIR}/docs/security-policy.md \
    --from-file=runbook.md=${CAFE_APP_DIR}/docs/runbook.md

echo ""
echo "ConfigMaps created:"
oc get configmap -n mcp-servers --no-headers | grep cafe

=== Creating cafe-source-code ConfigMap ===
configmap/cafe-source-code created

=== Creating cafe-docs ConfigMap ===
configmap/cafe-docs created

ConfigMaps created:
cafe-docs                          5     1s
cafe-source-code                   12    3s


## 8. Verify Server 4 — Codebase Search (Semantic Code Search)

Semantic search over internal source code using sentence-transformers embeddings.
Indexes all Python files from the `cafe-order-system` demo app.

**Tools:** `search_code`, `get_file`, `list_files`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | Yes (image includes pre-downloaded model) |

In [10]:
%%bash
DEPLOY="mcp-codebase-search"
NS="mcp-servers"

if oc get deployment/$DEPLOY -n $NS &>/dev/null; then
    STATUS=$(oc get deployment/$DEPLOY -n $NS -o jsonpath='{.status.conditions[?(@.type=="Available")].status}')
    if [ "$STATUS" = "True" ]; then
        echo "$DEPLOY is already running."
        echo ""
        echo "Route URL:"
        oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
        echo ""
        exit 0
    fi
    echo "$DEPLOY exists but is not ready (likely missing ConfigMap cafe-source-code)."
    echo "Applying manifest..."
fi

oc apply -f manifests/05-codebase-search.yaml

echo ""
echo "Waiting for pod (embedding model loading takes ~30s)..."
oc wait --for=condition=available deployment/$DEPLOY -n $NS --timeout=300s 2>/dev/null \
    && echo "Codebase Search ready" \
    || echo "Pod still starting... check: oc logs deploy/$DEPLOY -n $NS"

echo ""
echo "Route URL:"
oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
echo ""

deployment.apps/mcp-codebase-search created
service/mcp-codebase-search created
route.route.openshift.io/mcp-codebase-search created

Waiting for pod (embedding model loading takes ~30s)...
deployment.apps/mcp-codebase-search condition met
Codebase Search ready

Route URL:
https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp


## 9. Verify Server 5 — Repo Docs (Documentation Q&A)

Semantic search over internal documentation (architecture, API guides, runbooks, security policies).
Uses sentence-transformers for local embedding — no external LLM calls.

**Tools:** `search_docs`, `list_docs`

| Environment | Works? |
|-------------|--------|
| Internet access | Yes |
| Air-gapped | Yes (image includes pre-downloaded model) |

In [11]:
%%bash
DEPLOY="mcp-repo-docs"
NS="mcp-servers"

if oc get deployment/$DEPLOY -n $NS &>/dev/null; then
    STATUS=$(oc get deployment/$DEPLOY -n $NS -o jsonpath='{.status.conditions[?(@.type=="Available")].status}')
    if [ "$STATUS" = "True" ]; then
        echo "$DEPLOY is already running."
        echo ""
        echo "Route URL:"
        oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
        echo ""
        exit 0
    fi
    echo "$DEPLOY exists but is not ready (likely missing ConfigMap cafe-docs)."
    echo "Applying manifest..."
fi

oc apply -f manifests/06-repo-docs.yaml

echo ""
echo "Waiting for pod (embedding model loading takes ~30s)..."
oc wait --for=condition=available deployment/$DEPLOY -n $NS --timeout=300s 2>/dev/null \
    && echo "Repo Docs ready" \
    || echo "Pod still starting... check: oc logs deploy/$DEPLOY -n $NS"

echo ""
echo "Route URL:"
oc get route $DEPLOY -n $NS -o jsonpath='https://{.spec.host}/mcp'
echo ""

deployment.apps/mcp-repo-docs created
service/mcp-repo-docs created
route.route.openshift.io/mcp-repo-docs created

Waiting for pod (embedding model loading takes ~30s)...
deployment.apps/mcp-repo-docs condition met
Repo Docs ready

Route URL:
https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp


## 10. Verify All Servers

In [17]:
%%bash
echo "MCP Server Deployment Status"
echo "============================================================"
echo ""
echo "=== Pods ==="
oc get pods -n mcp-servers --sort-by=.metadata.name

echo ""
echo "=== Routes (MCP Endpoints) ==="
echo ""
printf "%-25s %-10s %s\n" "SERVER" "AIR-GAP" "ENDPOINT"
printf "%-25s %-10s %s\n" "-------" "-------" "--------"

for route in $(oc get routes -n mcp-servers -o jsonpath='{.items[*].metadata.name}'); do
    host=$(oc get route $route -n mcp-servers -o jsonpath='{.spec.host}')
    case $route in
        mcp-context7|mcp-searxng) airgap="No" ;;
        *) airgap="Yes" ;;
    esac
    printf "%-25s %-10s %s\n" "$route" "$airgap" "https://${host}/mcp"
done

MCP Server Deployment Status

=== Pods ===
NAME                                   READY   STATUS      RESTARTS   AGE
mcp-code-sandbox-6846b9dd4b-t8clh      1/1     Running     0          67m
mcp-codebase-search-1-build            0/1     Completed   0          66m
mcp-codebase-search-76647d957b-cb75t   1/1     Running     0          56m
mcp-context7-6c97f777f6-6bmb2          1/1     Running     0          67m
mcp-repo-docs-1-build                  0/1     Completed   0          61m
mcp-repo-docs-64f4ff965f-794rk         1/1     Running     0          55m
mcp-searxng-7d8db89f77-xq2bn           1/1     Running     1          3d4h

=== Routes (MCP Endpoints) ===

SERVER                    AIR-GAP    ENDPOINT
-------                   -------    --------
mcp-code-sandbox          Yes        https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp
mcp-codebase-search       Yes        https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox3031.op

In [18]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

print("MCP Health Check (all servers):")
print("=" * 60)

init_payload = json.dumps({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "healthcheck", "version": "1.0"}}
})

for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        url = f"https://{host}/mcp"
        r = subprocess.run(
            ["curl", "-sk", "-X", "POST",
             "-H", "Content-Type: application/json",
             "-H", "Accept: application/json, text/event-stream",
             "-d", init_payload,
             "-o", "/dev/null", "-w", "%{http_code}", "-m", "10", url],
            capture_output=True, text=True)
        code = r.stdout.strip()
        status = "PASS" if code == "200" else f"FAIL ({code})"
        print(f"  [{status}] {name}: {url}")

if not result.stdout.strip():
    print("  No routes found. Deploy servers first.")

MCP Health Check (all servers):
  [PASS] mcp-code-sandbox: https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp
  [PASS] mcp-codebase-search: https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp
  [PASS] mcp-context7: https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp
  [PASS] mcp-repo-docs: https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp
  [PASS] mcp-searxng: https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox3031.opentlc.com/mcp


## 11. Register in RHOAI Portal (GenAI Studio)

The RHOAI Dashboard shows MCP servers in **GenAI Studio** and **AI Hub → MCP Catalog**.
The portal reads from a `gen-ai-aa-mcp-servers` ConfigMap — it does not directly watch `MCPServerRegistration` CRDs.

> Without this ConfigMap, GenAI Studio shows "MCP servers ConfigMap may not be deployed yet".

In [14]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"

CTX7_PORT=$(oc get svc mcp-context7 -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "3001")
SXG_PORT=$(oc get svc mcp-searxng -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "8000")
SANDBOX_PORT=$(oc get svc mcp-code-sandbox -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "3005")
SEARCH_PORT=$(oc get svc mcp-codebase-search -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "8000")
DOCS_PORT=$(oc get svc mcp-repo-docs -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "8000")

oc apply -f - <<EOF
apiVersion: v1
kind: ConfigMap
metadata:
  name: gen-ai-aa-mcp-servers
  namespace: redhat-ods-applications
  labels:
    app: gen-ai
    app.kubernetes.io/part-of: gen-ai-studio
data:
  context7: |
    {
      "name": "context7",
      "displayName": "Context7 - Library Documentation",
      "description": "Provides up-to-date documentation for popular libraries and frameworks",
      "url": "http://mcp-context7.${MCP_NS}.svc.cluster.local:${CTX7_PORT}/mcp",
      "transport": "streamable-http",
      "category": "documentation"
    }
  searxng: |
    {
      "name": "searxng",
      "displayName": "SearXNG - Web Search",
      "description": "Web search capability for real-time information retrieval",
      "url": "http://mcp-searxng.${MCP_NS}.svc.cluster.local:${SXG_PORT}/mcp",
      "transport": "streamable-http",
      "category": "search"
    }
  code-sandbox: |
    {
      "name": "code-sandbox",
      "displayName": "Code Sandbox - Execution Environment",
      "description": "Safe code execution sandbox for testing and validation",
      "url": "http://mcp-code-sandbox.${MCP_NS}.svc.cluster.local:${SANDBOX_PORT}/mcp",
      "transport": "sse",
      "category": "development"
    }
  codebase-search: |
    {
      "name": "codebase-search",
      "displayName": "Codebase Search - Internal Code RAG",
      "description": "Semantic search over internal application source code",
      "url": "http://mcp-codebase-search.${MCP_NS}.svc.cluster.local:${SEARCH_PORT}/mcp",
      "transport": "streamable-http",
      "category": "development"
    }
  repo-docs: |
    {
      "name": "repo-docs",
      "displayName": "Repo Docs - Internal Documentation Q&A",
      "description": "Semantic search over internal documentation and guides",
      "url": "http://mcp-repo-docs.${MCP_NS}.svc.cluster.local:${DOCS_PORT}/mcp",
      "transport": "streamable-http",
      "category": "documentation"
    }
EOF

echo ""
echo "✅ ConfigMap created. Check RHOAI Dashboard:"
echo "   → GenAI Studio → select namespace → MCP Servers tab"
echo "   → AI Hub → MCP Catalog"

configmap/gen-ai-aa-mcp-servers configured

✅ ConfigMap created. Check RHOAI Dashboard:
   → GenAI Studio → select namespace → MCP Servers tab
   → AI Hub → MCP Catalog


## Summary

| Server | Type | Air-gapped | Key Tools |
|--------|------|:----------:|-----------|
| Context7 | External API | No | `resolve-library-id`, `get-library-docs` |
| SearXNG | Self-hosted meta-engine | No | `search-web`, `fetch-web` |
| Code Sandbox | Local runtime | Yes | `execute_code`, `read_file`, `write_file` |
| Codebase Search | AI Embeddings | Yes | `search_code`, `get_file`, `list_files` |
| Repo Docs | AI Embeddings | Yes | `search_docs`, `list_docs` |

**Air-gapped deployment**: Code Sandbox + Codebase Search + Repo Docs all work without internet.

## Next Steps

- `3_connect_ide_clients.ipynb` — Configure your IDE to connect to these MCP servers
- `../3_basic_run/` — Run the coding assistant with all tools enabled